In [2]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from qutip_qip.operations import (berkeley, cnot, cphase, csign, fredkin,
                                  gate_sequence_product, globalphase, iswap,
                                  molmer_sorensen, phasegate, qrot, rx, ry, rz,
                                  snot, sqrtiswap, sqrtnot, sqrtswap, swap,
                                  swapalpha, toffoli)
from qutip import gates

In [3]:
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]])
cnot_gate = np.array([[1, 0, 0, 0],
                      [0, 1, 0, 0],
                      [0, 0, 0, 1],
                      [0, 0, 1, 0]])
cz_gate = np.array([[1, 0, 0, 0],
                    [0, 1, 0, 0],
                    [0, 0, 1, 0],
                    [0, 0, 0, -1]])

In [ ]:
from numpy import size


def string_to_state(s):
    """
    Convert string to state, takes 0,1,+,-
    """
    states = []
    for i in s:
        if i == "0":
            states.append(np.array([[1], [0]]))
        elif i == "1":
            states.append(np.array([[0], [1]]))
        elif i == "+":
            states.append(np.array([[1], [1]]) / np.sqrt(2))
        elif i == "-":
            states.append(np.array([[1], [-1]]) / np.sqrt(2))
    for state in states:
        if 'result' in locals():
            result = np.kron(result, state)
        else:
            result = state
    return result
def Hgate(n):
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    step = H
    for i in range(n-1):
        step = np.kron(step, H)
    return step

def Rx(theta, size):
    gate = np.array([[np.cos(theta/2), -1j*np.sin(theta/2)], [-1j*np.sin(theta/2), np.cos(theta/2)]])
    step = gate
    for i in range(size-1):
        step = np.kron(step, gate)
    return step

def Ry(theta, size):
    gate = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])
    step = gate
    for i in range(size-1):
        step = np.kron(step, gate)
    return step

def Rz(theta, size):
    gate = np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]])
    step = gate
    for i in range(size-1):
        step = np.kron(step, gate)
    return step

def H_target(state, target, size):
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    I = np.eye(2)
    factors = [H if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def X_target(state, target, size):
    X = np.array([[0, 1], [1, 0]])
    I = np.eye(2)
    factors = [X if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def Z_target(state, target, size):
    Z = np.array([[1, 0], [0, -1]])
    I = np.eye(2)
    factors = [Z if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def CNOT(control, target, size):
    """
    (|0><0|)_c ⊗ I_rest + (|1><1|)_c ⊗ X_t ⊗ I_rest
    """
    P0 = np.array([[1, 0], [0, 0]]) #checks if qbit is 0, if yes, keep
    P1 = np.array([[0, 0], [0, 1]]) #checks if qbit is 1, if yes, apply X to target
    I  = np.eye(2)
    X  = np.array([[0, 1], [1, 0]])
    # Term 1: Control is |0> -> apply Identity to target
    term0 = [P0 if i == control else I for i in range(size)]
    
    # Term 2: Control is |1> -> apply X to target
    term1 = [X if i == target else (P1 if i == control else I) for i in range(size)]

    #compute tensor
    op0 = term0[0]
    op1 = term1[0]
    for i in range(1, size):
        op0 = np.kron(op0, term0[i])
        op1 = np.kron(op1, term1[i])
    return op0 + op1

def CZ(control, target, size):
    """
    (|0><0|)_c ⊗ I_rest + (|1><1|)_c ⊗ Z_t ⊗ I_rest
    """
    gate = np.eye(2**size)
    P0 = np.array([[1, 0], [0, 0]]) #checks if qbit is 0, if yes, keep
    P1 = np.array([[0, 0], [0, 1]]) #checks if qbit is 1, if yes, apply Z to target
    I  = np.eye(2)
    Z  = np.array([[1, 0], [0, -1]])
    #control is |0> -> apply Identity to target
    term0 = [P0 if i == control else I for i in range(size)]
    # Control is |1> -> apply Z to target
    term1 = [Z if i == target else (P1 if i == control else I) for i in range(size)]
    #compute tensor
    op0 = term0[0]
    op1 = term1[0]
    for i in range(1, size):
        op0 = np.kron(op0, term0[i])
        op1 = np.kron(op1, term1[i])
    return op0 + op1

def measurement(state, qbit):
    size = int(np.log2(state.shape[0]))
    #extract the state of the qubit to be measured
    P0 = np.array([[1, 0], [0, 0]]) # |0><0|
    P1 = np.array([[0, 0], [0, 1]]) # |1><1|
    I  = np.eye(2)
    term0= [P0 if i == qbit else I for i in range(size)]
    term1= [P1 if i == qbit else I for i in range(size)]
    #compute tensor
    op0 = term0[0]
    op1 = term1[0]
    for i in range(1, size):
        op0 = np.kron(op0, term0[i])
        op1 = np.kron(op1, term1[i])
    #probabilities
    p0 = np.real(np.conj(state.T) @ op0 @ state)[0, 0] # prob of measuring |0>
    p1 = np.real(np.conj(state.T) @ op1 @ state)[0, 0] # prob of measuring |1>
    #randomly choose outcome
    outcome = np.random.choice([0, 1], p=[p0, p1])
    #collapse state
    if outcome == 0:
        new_state = op0 @ state / np.sqrt(p0)
    else:
        new_state = op1 @ state / np.sqrt(p1)
    return outcome, new_state, [p0, p1]

def state_teleportation(message, target):
    #initialize states, msg (0), resource (1), target (2)
    size = 3
    init = string_to_state("00")
    init = np.kron(init, message)
    all_init = init
    #H on 2
    init = H_target(init, 1, 3)
    init = CNOT(1,2,3)
    init = CNOT(0,1,3)
    init = H_target(init, 0, 3)
    measure_0_outcome, init, _ = measurement(init, 0)
    measure_1_outcome, init, _ = measurement(init, 1)
    if measure_0_outcome == 1: #if 1 then apply X to target
        init = X_target(init, 2, 3)
    if measure_1_outcome == 1: #if 1 then apply Z to target
        init = Z_target(init, 2, 3)
    final_state = init
    return final_state



In [22]:
#test for H_target
size = 3
state = string_to_state("000")
#H on 2
state = H_target(state, 1, size)
print(state)

[[0.70710678]
 [0.        ]
 [0.70710678]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]]


In [17]:
state= string_to_state("1+1")
qbit_to_measure = 1
outcome, new_state, probabilities = measurement(state, qbit_to_measure)
print(state)
print(f"Measurement outcome of qubit {qbit_to_measure}: {outcome}")
print(f"New state after measurement: {new_state}")
print(f"Probabilities: P(0) = {probabilities[0]}, P(1) = {probabilities[1]}")

[[0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.70710678]
 [0.        ]
 [0.70710678]]
Measurement outcome of qubit 1: 1
New state after measurement: [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]]
Probabilities: P(0) = 0.4999999999999999, P(1) = 0.4999999999999999
